# Interactive IGV Viewer — HPRC R2 PanGenie GEUVADIS eQTL
**Author:** Andrew Blair  
**Environment:** `plotly_dashbio_igv` (see `jupyterlab-nbi-plotly_dashbio-igv.yml`)

---

## Overview

This notebook launches an interactive [IGV.js](https://github.com/igvteam/igv.js) genome browser
inside JupyterLab using [Dash Bio](https://github.com/plotly/dash-bio) and a lightweight Flask
byte-range file server. All genomic data is served directly from the HPC filesystem —
no external hosting or data transfer required.

### Data
| Layer | Dataset | Details |
|---|---|---|
| Reference | GCA_000001405.15 GRCh38 no alt analysis set | NCBI GRCh38 primary assembly |
| Annotation | Gencode v47 | Gene models, GRCh38 coordinates |
| Variants | HPRC v2.0 MC graph, PanGenie genotypes | 397 GEUVADIS samples, biallelic + multiallelic |

### How it works
1. A **Flask server** registers a `/igv/<filename>` route on the Dash app
2. IGV.js makes **HTTP byte-range requests** to fetch only the data at the current locus
3. Large files (VCF.gz, FASTA) are never fully loaded — only the requested bytes are read
4. The viewer renders inline in the cell output

---

**To use:** Run all cells top-to-bottom (`Shift+Enter`) or **Kernel → Restart & Run All**

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
# Standard library
import os
import logging
import signal
import subprocess

# Flask — byte-range file server
from flask import request, Response, stream_with_context

# Dash — web app framework embedding IGV.js
from dash import Dash, html
import dash_bio as dashbio

# Suppress Werkzeug's per-request logs to prevent Jupyter output buffer overflow
logging.getLogger("werkzeug").setLevel(logging.ERROR)

## Configuration

Edit the variables below to adapt this notebook to a different dataset or port.

- **`PORT`** — each notebook must use a unique port to avoid conflicts on shared nodes
- **`ASSET_DIR`** — directory containing all genomic files (FASTA, VCF.gz, GTF, etc.)
- **`CHUNK_SIZE`** — size of streaming chunks for full-file requests (index files); 8MB is a safe default

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Port for the Dash/Flask server — must be unique per running notebook on this node
PORT = 8000

# Directory containing all indexed genomic files served to IGV.js
# Place files here (or symlink them) — any file in this directory is accessible
# via /igv/<filename>
ASSET_DIR = "/private/groups/patenlab/apblair/nbi_igv/assets/"

# Chunk size for streaming full files (e.g. index files fetched without Range header)
CHUNK_SIZE = 8 * 1024 * 1024  # 8 MB

# ── Port cleanup ───────────────────────────────────────────────────────────────
# Kill any leftover process on this port before starting (e.g. from a previous run)
result = subprocess.run(["lsof", "-ti", f":{PORT}"], capture_output=True, text=True)
pids = result.stdout.strip().split()
for pid in pids:
    os.kill(int(pid), signal.SIGKILL)
    print(f"Killed PID {pid} on port {PORT}")
if not pids:
    print(f"Port {PORT} is free")

## Dash App + Byte-Range File Server

The Dash app embeds IGV.js as an inline component. A custom Flask route
(`/igv/<filename>`) intercepts all IGV.js data requests and serves files
directly from `ASSET_DIR` on the HPC filesystem.

### Why byte-range serving?
IGV.js uses [HTTP Range requests](https://developer.mozilla.org/en-US/docs/Web/HTTP/Range_requests)
to fetch only the bytes it needs at the current locus. For a 30GB VCF.gz, IGV.js
will request ~10–100KB per locus query — the full file is never loaded into memory.

The server handles two request types:
- **`Range: bytes=X-Y`** → reads exactly `Y-X+1` bytes at offset `X`, returns `206 Partial Content`
- **No Range header** → used for small index files (`.tbi`, `.fai`); streams in 8MB chunks

In [ ]:
# ── Dash App ──────────────────────────────────────────────────────────────────
app = Dash(
    __name__,
    assets_folder=ASSET_DIR,
    assets_url_path="/assets"
)

# ── Byte-Range File Server ─────────────────────────────────────────────────────
# Registers /igv/<filename> on the Flask server underlying the Dash app.
# IGV.js will call this route for every genomic file (FASTA, VCF.gz, GTF, etc.)
@app.server.route('/igv/<path:filename>')
def serve_igv_file(filename):
    file_path = os.path.join(ASSET_DIR, filename)

    if not os.path.isfile(file_path):
        return "File not found", 404

    size = os.path.getsize(file_path)
    range_header = request.headers.get("Range")

    if range_header:
        # Partial content — IGV.js standard for data files (VCF.gz, FASTA, CRAM)
        start, end_str = range_header.replace("bytes=", "").split("-")
        start = int(start)
        end = int(end_str) if end_str else size - 1
        length = end - start + 1
        with open(file_path, "rb") as f:
            f.seek(start)
            data = f.read(length)
        resp = Response(data, 206, mimetype="application/octet-stream")
        resp.headers["Content-Range"] = f"bytes {start}-{end}/{size}"
        resp.headers["Accept-Ranges"] = "bytes"
        resp.headers["Content-Length"] = str(length)
        return resp
    else:
        # Full file — stream in chunks to avoid loading large files into memory
        # Typically used for index files (.tbi, .fai, .crai) fetched on load
        def generate():
            with open(file_path, "rb") as f:
                while chunk := f.read(CHUNK_SIZE):
                    yield chunk
        resp = Response(stream_with_context(generate()), 200,
                        mimetype="application/octet-stream")
        resp.headers["Accept-Ranges"] = "bytes"
        resp.headers["Content-Length"] = str(size)
        return resp

## IGV Layout

Defines the genome reference and tracks loaded into the viewer.

### Reference genome
The reference must be served as:
- Uncompressed FASTA + `.fai` index, **or**
- bgzipped FASTA + `.fai` + `.gzi` index

> ⚠️ **Known limitation:** `dash-bio==1.0.2` bundles an IGV.js version that does not
> support bgzipped FASTA + `.gzi`. Use uncompressed FASTA + `.fai` only.

### Tracks
Each track entry requires at minimum:

| Key | Description |
|---|---|
| `name` | Label shown in the IGV track header |
| `type` | `"annotation"`, `"variant"`, `"alignment"`, or `"wig"` |
| `format` | File format: `"gtf"`, `"vcf"`, `"bam"`, `"cram"`, `"bedgraph"`, etc. |
| `url` | `/igv/<filename>` — file must exist in `ASSET_DIR` |
| `indexURL` | `/igv/<filename>.tbi` or `.bai` / `.crai` |
| `displayMode` | `"EXPANDED"`, `"SQUISHED"`, or `"COLLAPSED"` |

### Locus
Set the initial view. Accepts chromosome, region, or gene name:
```python
locus = "chr15"                        # whole chromosome
locus = "chr15:50,000,000-51,000,000" # specific region
locus = "BRCA2"                        # gene name (requires annotation track)
```

In [ ]:
# ── IGV Layout ────────────────────────────────────────────────────────────────
app.layout = html.Div(
    style={"backgroundColor": "white"},
    children=[
        dashbio.Igv(
            id="igv-viewer",
            style={"backgroundColor": "white"},

            # ── Reference Genome ──────────────────────────────────────────────
            # GCA_000001405.15 — GRCh38 no alt analysis set (NCBI)
            # Served as uncompressed FASTA + FAI index (see known limitation above)
            genome={
                "id": "GRCh38",
                "name": "GCA_000001405.15 GRCh38 no alt analysis set",
                "fastaURL": "/igv/GCA_000001405.15_GRCh38_no_alt_analysis_set.fasta",
                "indexURL": "/igv/GCA_000001405.15_GRCh38_no_alt_analysis_set.fasta.fai",
                "showSVGButton": True,   # enables SVG export button in IGV toolbar
            },

            # ── Tracks ────────────────────────────────────────────────────────
            # Add or remove track entries here to customise the viewer.
            # Files must be in ASSET_DIR (or symlinked there).
            tracks=[

                # Gene annotation — Gencode v47 (GRCh38)
                # Sorted, BGZF-compressed GTF with tabix index
                {
                    "name": "Gencode v47",
                    "type": "annotation",
                    "format": "gtf",
                    "displayMode": "EXPANDED",
                    "url": "/igv/gencode.v47.annotation.gtf.bgz",
                    "indexURL": "/igv/gencode.v47.annotation.gtf.bgz.tbi"
                },

                # Variant calls — HPRC v2.0 MC graph, PanGenie genotypes
                # 397 GEUVADIS samples; converted from BCF → VCF.gz for IGV.js compatibility
                # displayMode SQUISHED recommended for large multi-sample cohorts
                {
                    "name": "HPRC v2.0 GRCh38 Cohort (397)",
                    "type": "variant",
                    "format": "vcf",
                    "displayMode": "SQUISHED",
                    "url": "/igv/geuvadis_hprc_v2.0_mc_grch38_397.vcf.gz",
                    "indexURL": "/igv/geuvadis_hprc_v2.0_mc_grch38_397.vcf.gz.tbi"
                },

            ],

            # ── Initial locus ─────────────────────────────────────────────────
            # Edit to change the region displayed on startup
            locus="chr15",

            # Minimum number of bases before nucleotide-level view is shown
            minimumBases=2
        )
    ]
)

# ── Launch ────────────────────────────────────────────────────────────────────
# mode="inline" renders the viewer in the cell output (JupyterLab only)
# dev_tools_silence_routes_logging=True prevents route logs flooding the output buffer
app.run(mode="inline", port=PORT, height=600,
        dev_tools_silence_routes_logging=True)